# Pre/Post S and I Energy: Data Processing and Min-Max Normalization

This notebook reads the source CSV files, assigns energy valleys to behavioral states for each mouse and phase, and exports the raw and normalized results.

**Normalization unit:** one mouse and one behavioral state. Each group contains six pre observations and six post observations. The same minimum and maximum are applied to all 12 observations. S (struggle) and I (immobility) are normalized separately.

\[
x_{\mathrm{norm}}=\frac{x-x_{\min}}{x_{\max}-x_{\min}}
\]

**Energy and behavioral mapping:** use `Final_energy_valley1` and `Final_energy_valley2`. In each phase's `posthoc_valley_behavior_mapping.csv`, `valley0_label` identifies the state for `Final_energy_valley1`, and `valley1_label` identifies the state for `Final_energy_valley2`. The assignment can change between pre and post.

**Time interpretation:** preserve the source index (0-5) and display it as `minute_ordinal` (1-6). Related analysis code interprets these observations as cumulative estimates through minutes 1-6. This notebook does not recalculate energy for non-overlapping one-minute intervals.

CSV files use UTF-8 with BOM and 17 significant digits. Notebook previews display six decimal places without rounding the underlying data. Source file references in CSV outputs are relative to `SOURCE_ROOT`.

Select a Python 3 kernel and use **Restart Kernel and Run All**. Required packages: `pandas`, `numpy`, and `IPython`.

## 1. Configuration and helpers

Replace the path placeholders with your input and output folders. The input root contains subfolders named `<mouse ID> pre` and `<mouse ID> post`; each contains `energy_landscape_results_m.csv` and `posthoc_valley_behavior_mapping.csv`.

Replace each `your mouse ID XXX` entry with a distinct actual mouse ID in the intended order. The repeated placeholders must be configured before running, and each ID must match its folder names.

Historical outputs and execution metadata have been cleared from this anonymized copy. Validation remains limited to the main data requirements and one read-back of the final detailed CSV. Each processing step still displays its output when the notebook is run.


In [ ]:
from pathlib import Path
from datetime import datetime
import platform
import re
import numpy as np
import pandas as pd
from IPython.display import display

SOURCE_ROOT = Path("path to your SOURCE_ROOT fold")  # Input folder: <mouse ID> pre/post subfolders containing energy_landscape_results_m.csv and posthoc_valley_behavior_mapping.csv.
OUTPUT_DIR = Path("path to your OUTPUT_DIR fold")  # Output folder for raw/normalized energy CSVs, summaries, validation results, and manifests.
# Replace each XXX with a distinct mouse ID in the intended order; use the same ID for pre and post.
MOUSE_IDS = [
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
    'your mouse ID XXX',
]
PHASES = ['pre', 'post']
SOURCE_INDICES = list(range(6))
RECORD_KEY = ['mouse_id', 'phase', 'minute_ordinal']
RUN_TIME = datetime.now().astimezone().isoformat(timespec='seconds')
TOLERANCE = 1e-12

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 18)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda value: f'{value:.6f}')
validation_records = []
export_records = []

def validate(condition, step, description):
    passed = bool(condition)
    validation_records.append({'step': step, 'check': description,
                               'status': 'PASS' if passed else 'FAIL'})
    if not passed:
        raise ValueError(f'Step {step}: {description}')

def show_validation(step):
    display(pd.DataFrame([record for record in validation_records if record['step'] == step]))

def save_csv(frame, filename):
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False, encoding='utf-8-sig', float_format='%.17g')
    export_records.append({'filename': filename, 'rows': len(frame), 'columns': len(frame.columns),
                           'bytes': path.stat().st_size, 'path': str(path), 'run_time': RUN_TIME})
    print(f'Saved {filename}: {len(frame)} rows, {len(frame.columns)} columns')
    return path

def sort_records(frame):
    return (frame.assign(_mouse_order=frame['mouse_id'].map({mouse:i for i,mouse in enumerate(MOUSE_IDS)}),
                         _phase_order=frame['phase'].map({'pre':0, 'post':1}))
            .sort_values(['_mouse_order', '_phase_order', 'minute_ordinal'])
            .drop(columns=['_mouse_order', '_phase_order']).reset_index(drop=True))

print('Run time:', RUN_TIME)
print('Python / pandas / numpy:', platform.python_version(), pd.__version__, np.__version__)
print('Output directory:', OUTPUT_DIR)
print(f'Cohort: {len(MOUSE_IDS)} mice; phases: {", ".join(PHASES)}')

## 2. Locate the pre and post inputs

Select the configured mice and phases. Rescue folders and other mice are outside this analysis. Each phase uses one energy file and one behavioral mapping file. Paths in the input manifest are relative to `SOURCE_ROOT`.

In [ ]:
phase_folders = {}
for folder in SOURCE_ROOT.iterdir():
    if folder.is_dir():
        match = re.fullmatch(r'(.+) (pre|post)', folder.name)
        if match:
            phase_folders[(match.group(1), match.group(2))] = folder

required_pairs = {(mouse, phase) for mouse in MOUSE_IDS for phase in PHASES}
validate(required_pairs <= phase_folders.keys(), 2,
         'Every configured mouse has both pre and post folders')

input_rows = []
for mouse in MOUSE_IDS:
    for phase in PHASES:
        folder = phase_folders[(mouse, phase)]
        input_rows.append({
            'mouse_id': mouse, 'phase': phase,
            'energy_file': (folder / 'energy_landscape_results_m.csv').relative_to(SOURCE_ROOT).as_posix(),
            'mapping_file': (folder / 'posthoc_valley_behavior_mapping.csv').relative_to(SOURCE_ROOT).as_posix()})
input_manifest = pd.DataFrame(input_rows)
save_csv(input_manifest, '01_input_manifest.csv')
display(input_manifest)
show_validation(2)

## 3. Read the energy records and behavioral mappings

Preserve all 13 source energy and valley-coordinate fields. Rename the source index column to `csv_index` and set `minute_ordinal = csv_index + 1`.

Check that each mouse-phase pair contains indices 0-5, the source numbers are finite, and each mapping contains exactly one struggle label and one immobility label.

In [ ]:
ENERGY_COLUMNS = [
    'valley1_F_cluster1', 'valley1_F_cluster2', 'valley2_F_cluster1', 'valley2_F_cluster2',
    'energy_valley1', 'energy_valley2', 'energy_barrier',
    'Final_valley1_F_cluster1', 'Final_valley1_F_cluster2',
    'Final_valley2_F_cluster1', 'Final_valley2_F_cluster2',
    'Final_energy_valley1', 'Final_energy_valley2']

energy_parts, mapping_parts = [], []
for item in input_manifest.itertuples(index=False):
    energy = pd.read_csv(SOURCE_ROOT / item.energy_file, encoding='utf-8-sig', float_precision='round_trip')
    energy = energy.rename(columns={'Unnamed: 0': 'csv_index'})[['csv_index'] + ENERGY_COLUMNS]
    energy.insert(0, 'minute_ordinal', energy['csv_index'] + 1)
    energy.insert(0, 'phase', item.phase)
    energy.insert(0, 'mouse_id', item.mouse_id)
    energy['energy_file'] = item.energy_file
    energy_parts.append(energy)

    mapping = pd.read_csv(SOURCE_ROOT / item.mapping_file, encoding='utf-8-sig', float_precision='round_trip')
    mapping.insert(0, 'phase', item.phase)
    mapping.insert(0, 'mouse_id', item.mouse_id)
    mapping['mapping_file'] = item.mapping_file
    mapping_parts.append(mapping)

raw_energy = sort_records(pd.concat(energy_parts, ignore_index=True))
behavior_mapping = pd.concat(mapping_parts, ignore_index=True)
index_sequences = raw_energy.groupby(['mouse_id','phase'])['csv_index'].agg(list)
validate(index_sequences.map(lambda indices: indices == SOURCE_INDICES).all(), 3,
         'Each mouse-phase pair contains source indices 0-5 exactly once')
validate(np.isfinite(raw_energy[ENERGY_COLUMNS].to_numpy(dtype=float)).all(), 3,
         'All source energy and coordinate values are finite')
label_pairs = behavior_mapping[['valley0_label','valley1_label']].apply(
    lambda row: sorted(row.tolist()) == ['immobility','struggle'], axis=1)
validate(behavior_mapping.groupby(['mouse_id','phase']).size().eq(1).all()
         and len(behavior_mapping) == len(input_manifest) and label_pairs.all(), 3,
         'Each mouse-phase pair has one valid struggle/immobility mapping')

behavior_mapping['S_valley'] = np.where(behavior_mapping['valley0_label'].eq('struggle'), 1, 2)
behavior_mapping['I_valley'] = 3 - behavior_mapping['S_valley']
behavior_mapping = behavior_mapping[[column for column in behavior_mapping.columns if column != 'mapping_file']
                                    + ['mapping_file']]

window_exceptions = behavior_mapping.loc[behavior_mapping['valid_window_count'] != 120,
                                         ['mouse_id','phase','valid_window_count']]
if not window_exceptions.empty:
    print('Behavioral window counts other than 120; all six source energy observations are retained:')
    display(window_exceptions)

save_csv(raw_energy, '02_raw_energy_records.csv')
save_csv(behavior_mapping, '03_behavior_valley_mapping.csv')
display(raw_energy.head(12))
display(behavior_mapping[['mouse_id','phase','valley0_label','valley1_label',
                          'S_valley','I_valley','valid_window_count']])
show_validation(3)

## 4. Assign S and I energy for each mouse and phase

Join the behavioral mapping to the energy records using `mouse_id` and `phase`. Select the corresponding Final energy column separately for each phase, then sort by mouse, pre/post phase, and minute ordinal.

In [ ]:
mapped_energy = raw_energy.merge(
    behavior_mapping[['mouse_id','phase','S_valley','I_valley','mapping_file']],
    on=['mouse_id','phase'], how='left', validate='many_to_one', indicator=True)
validate(mapped_energy['_merge'].eq('both').all(), 4,
         'Every energy record matches its mouse-phase behavioral mapping')
mapped_energy['S_energy'] = np.where(mapped_energy['S_valley'].eq(1),
                                    mapped_energy['Final_energy_valley1'], mapped_energy['Final_energy_valley2'])
mapped_energy['I_energy'] = np.where(mapped_energy['I_valley'].eq(1),
                                    mapped_energy['Final_energy_valley1'], mapped_energy['Final_energy_valley2'])
si_energy = sort_records(mapped_energy[RECORD_KEY + ['csv_index','S_valley','I_valley','S_energy','I_energy',
                                                   'energy_file','mapping_file']])

phase_mapping = behavior_mapping.pivot(index='mouse_id', columns='phase', values='S_valley').reindex(MOUSE_IDS)
changed_mapping = phase_mapping.loc[phase_mapping['pre'] != phase_mapping['post']]
print(f'{len(changed_mapping)} mice have different S valley assignments in pre and post:')
display(changed_mapping)
save_csv(si_energy, '04_SI_energy_by_mouse.csv')
display(si_energy.head(12))
show_validation(4)

## 5. Normalize using combined pre/post bounds

Reshape the S/I values into one row per behavioral state and observation. Compute the minimum and maximum for each `mouse_id` and `state` across all 12 pre/post observations.

Validate group sizes and nonzero ranges, then check that normalized values lie in [0, 1] and each group reaches both endpoints.

In [ ]:
energy_long = si_energy.melt(id_vars=RECORD_KEY, value_vars=['S_energy','I_energy'],
                             var_name='state', value_name='energy')
energy_long['state'] = energy_long['state'].map({'S_energy':'S', 'I_energy':'I'})
minmax_parameters = (energy_long.groupby(['mouse_id','state'], sort=False)['energy']
                     .agg(n='size', minimum='min', maximum='max').reset_index())
minmax_parameters['span'] = minmax_parameters['maximum'] - minmax_parameters['minimum']
minmax_parameters = (
    minmax_parameters.assign(_mouse_order=minmax_parameters['mouse_id'].map({mouse:i for i,mouse in enumerate(MOUSE_IDS)}),
                             _state_order=minmax_parameters['state'].map({'S':0,'I':1}))
    .sort_values(['_mouse_order','_state_order']).drop(columns=['_mouse_order','_state_order']).reset_index(drop=True))
validate(len(minmax_parameters) == 2*len(MOUSE_IDS)
         and minmax_parameters['n'].eq(12).all() and minmax_parameters['span'].gt(0).all(), 5,
         'Every mouse-state group has 12 observations and a nonzero range')

normalized_long = energy_long.merge(minmax_parameters, on=['mouse_id','state'], validate='many_to_one')
normalized_long['normalized'] = ((normalized_long['energy'] - normalized_long['minimum'])
                                / normalized_long['span'])
validate(np.isfinite(normalized_long['normalized']).all()
         and normalized_long['normalized'].between(-TOLERANCE, 1+TOLERANCE).all(), 5,
         'All normalized values are finite and lie in [0, 1]')
normalized_bounds = normalized_long.groupby(['mouse_id','state'])['normalized'].agg(['min','max'])
validate(np.allclose(normalized_bounds['min'], 0, rtol=0, atol=TOLERANCE)
         and np.allclose(normalized_bounds['max'], 1, rtol=0, atol=TOLERANCE), 5,
         'Each normalized mouse-state group has minimum 0 and maximum 1')

normalized_wide = (normalized_long.pivot(index=RECORD_KEY, columns='state', values='normalized')
                   .rename(columns={'S':'S_norm','I':'I_norm'}).reset_index())
normalized_wide.columns.name = None
normalized_records = sort_records(si_energy.merge(normalized_wide, on=RECORD_KEY, validate='one_to_one'))
normalized_records = normalized_records[RECORD_KEY + ['S_energy','I_energy','S_norm','I_norm','S_valley','I_valley',
                                                     'csv_index','energy_file','mapping_file']]
save_csv(minmax_parameters, '05_minmax_parameters_pre_post.csv')
save_csv(normalized_records, '06_SI_normalized_by_mouse.csv')
display(minmax_parameters)
display(normalized_records.head(12))
show_validation(5)

## 6. Export raw and normalized summary matrices

Each matrix uses pre observations 1-6 followed by post observations 1-6. Mouse columns follow `MOUSE_IDS`. S and I are exported separately for both raw energy and normalized energy.

In [ ]:
summary_index = pd.MultiIndex.from_product([PHASES, range(1,7)], names=['phase','minute_ordinal'])
summary_tables = {}
summary_specs = [('S_energy','07_S_raw_summary.csv'), ('I_energy','08_I_raw_summary.csv'),
                 ('S_norm','09_S_normalized_summary.csv'), ('I_norm','10_I_normalized_summary.csv')]
for variable, filename in summary_specs:
    matrix = (normalized_records.pivot(index=['phase','minute_ordinal'], columns='mouse_id', values=variable)
              .reindex(index=summary_index, columns=MOUSE_IDS))
    matrix.columns.name = None
    summary_tables[variable] = matrix.reset_index()

validate(all(table[MOUSE_IDS].notna().all().all() for table in summary_tables.values()), 6,
         'All four summary matrices contain every mouse and observation')
for variable, filename in summary_specs:
    save_csv(summary_tables[variable], filename)

print('S normalized summary:')
display(summary_tables['S_norm'])
print('I normalized summary:')
display(summary_tables['I_norm'])
show_validation(6)

## 7. Verify the final detailed CSV

Read back the main result once and compare it with the in-memory table. IDs and source references remain strings, and floating-point values are parsed with round-trip precision.

In [ ]:
text_columns = {column: str for column in normalized_records.columns
                if not pd.api.types.is_numeric_dtype(normalized_records[column])}
reloaded_results = pd.read_csv(OUTPUT_DIR / '06_SI_normalized_by_mouse.csv', encoding='utf-8-sig',
                               dtype=text_columns, float_precision='round_trip', keep_default_na=False)
pd.testing.assert_frame_equal(normalized_records, reloaded_results, check_dtype=False, check_exact=True)
validate(True, 7, 'The final detailed CSV preserves all values, keys, and source references exactly')
display(reloaded_results.head(12))
show_validation(7)

## 8. Save the validation summary and output manifest

Primary results:

- `06_SI_normalized_by_mouse.csv`: raw and normalized S/I energy for every mouse, phase, and observation.
- `09_S_normalized_summary.csv`: normalized S matrix.
- `10_I_normalized_summary.csv`: normalized I matrix.
- `05_minmax_parameters_pre_post.csv`: the 24 mouse-state normalization ranges.

`11_validation_summary.csv` records the essential checks. `12_output_manifest.csv` lists the ten data CSV files and the validation summary, excluding the manifest itself.

All source file references are relative to `SOURCE_ROOT`. `minute_ordinal` is the displayed observation number; `csv_index` preserves the source index.

In [ ]:
validation_summary = pd.DataFrame(validation_records)
save_csv(validation_summary, '11_validation_summary.csv')
output_manifest = pd.DataFrame(export_records)
output_manifest.to_csv(OUTPUT_DIR / '12_output_manifest.csv', index=False, encoding='utf-8-sig')

display(validation_summary)
display(output_manifest[['filename','rows','columns','bytes']])
print(f'Completed: {len(MOUSE_IDS)} mice, {len(normalized_records)} records, '
      f'{len(normalized_long)} normalized values, and {len(minmax_parameters)} normalization groups.')
print(f'Validation: {len(validation_summary)} essential checks passed.')
print('Saved 12 CSV files to:', OUTPUT_DIR)